In [5]:
import re
from pathlib import Path

import pandas as pd

def load_movie_rating_stats() -> pd.DataFrame:
    """Load per-movie rating counts and average ratings from processed data."""

    return pd.read_csv("F:\MovieMate-AI\data\processed\movie_rating_stats.csv")


def merge_movies_with_ratings(
    movies: pd.DataFrame,
    rating_stats: pd.DataFrame,
) -> pd.DataFrame:
    """Join movie metadata with per-movie rating statistics.

    Movies without ratings are excluded because they cannot be ranked by
    rating-based popularity.
    """

    movies_with_ratings = movies.merge(
        rating_stats,
        on="movieId",
        how="inner",
        validate="one_to_one",
    )

    return movies_with_ratings
rating = load_movie_rating_stats()
movies = pd.read_csv("F:\MovieMate-AI\data\processed\movies_processed.csv")
df = merge_movies_with_ratings(movies , rating)
df.head(5)

,movieId,title,genres,tmdbId,release_year,clean_title,genres_text,genre_count,rating_count,average_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,1995.0,Toy Story,Adventure Animation Children Comedy Fantasy,5,57309,3.893708
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,1995.0,Jumanji,Adventure Children Fantasy,3,24228,3.251527
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,1995.0,Grumpier Old Men,Comedy Romance,2,11804,3.142028
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,1995.0,Waiting to Exhale,Comedy Drama Romance,3,2523,2.853547
4,5,Father of the Bride Part II (1995),Comedy,11862.0,1995.0,Father of the Bride Part II,Comedy,1,11714,3.058434


In [6]:
df["rating_count"].sum()

np.int64(25000095)

In [9]:
df["rating_count"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95]
)

count    59047.000000
mean       423.393144
std       2477.885821
min          1.000000
50%          6.000000
75%         36.000000
90%        413.000000
95%       1503.400000
max      81491.000000
Name: rating_count, dtype: float64

In [18]:
global_mean_rating = (
        (
            df["average_rating"]
            * df["rating_count"]
        ).sum()
        / df["rating_count"].sum()
    )
minimum_rating_count = df["rating_count"].quantile(0.90)

vote_weight = (
        df["rating_count"]
        / (df["rating_count"] + minimum_rating_count)
    )
df["popularity_score"] = (
        vote_weight * df["average_rating"]
        + (1 - vote_weight) * global_mean_rating
    )

In [19]:
df.head()

,movieId,title,genres,tmdbId,release_year,clean_title,genres_text,genre_count,rating_count,average_rating,popularity_score
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,1995.0,Toy Story,Adventure Animation Children Comedy Fantasy,5,57309,3.893708,3.891133
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,1995.0,Jumanji,Adventure Children Fantasy,3,24228,3.251527,3.256259
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,1995.0,Grumpier Old Men,Comedy Romance,2,11804,3.142028,3.155274
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,1995.0,Waiting to Exhale,Comedy Drama Romance,3,2523,2.853547,2.949245
4,5,Father of the Bride Part II (1995),Comedy,11862.0,1995.0,Father of the Bride Part II,Comedy,1,11714,3.058434,3.074625


In [28]:
df.shape

(59047, 11)

In [32]:
df[df["genres"].isna()]

,movieId,title,genres,tmdbId,release_year,clean_title,genres_text,genre_count,rating_count,average_rating,popularity_score
15865,83773,Away with Words (San tiao ren) (1999),NaN,24403.0,1999.0,Away with Words (San tiao ren),NaN,0,2,1.750000,3.525258
16044,84768,Glitterbug (1994),NaN,140527.0,1994.0,Glitterbug,NaN,0,3,2.833333,3.528803
16333,86493,"Age of the Earth, The (A Idade da Terra) (1980)",NaN,76816.0,1980.0,"Age of the Earth, The (A Idade da Terra)",NaN,0,2,2.000000,3.526462
16472,87061,Trails (Veredas) (1978),NaN,83230.0,1978.0,Trails (Veredas),NaN,0,4,4.000000,3.538326
17377,91246,Milky Way (Tejút) (2007),NaN,125161.0,2007.0,Milky Way (Tejút),NaN,0,5,3.300000,3.531057
...,...,...,...,...,...,...,...,...,...,...,...
59024,209101,Hua yang de nian hua (2001),NaN,187548.0,2001.0,Hua yang de nian hua,NaN,0,1,2.000000,3.530149
59025,209103,Tsar Ivan the Terrible (1991),NaN,389979.0,1991.0,Tsar Ivan the Terrible,NaN,0,1,4.000000,3.534980
59031,209133,The Riot and the Dance (2018),NaN,519334.0,2018.0,The Riot and the Dance,NaN,0,2,2.500000,3.528872
59039,209151,Mao Zedong 1949 (2019),NaN,622831.0,2019.0,Mao Zedong 1949,NaN,0,1,3.500000,3.533773


In [47]:
genre = "Drama"
genre = genre.casefold()
movie_with_genre = df[
    df["genres"].fillna("").str.casefold().str.contains(genre)
]
if movie_with_genre.empty:
    print(ValueError("genre not found"))
movie_with_genre = movie_with_genre.sort_values(by="popularity_score" , ascending=False)
movie_with_genre = movie_with_genre[
    ["clean_title",
"release_year",
"genres",
"rating_count",
"average_rating",
"popularity_score"]
]
movie_with_genre.head(10)

,clean_title,release_year,genres,rating_count,average_rating,popularity_score
314,"Shawshank Redemption, The",1994.0,Crime|Drama,81482,4.413576,4.409140
840,"Godfather, The",1972.0,Crime|Drama,52498,4.324336,4.318166
1190,"Godfather: Part II, The",1974.0,Crime|Drama,34188,4.261759,4.253070
522,Schindler's List,1993.0,Drama|War,60411,4.247579,4.242733
1930,Seven Samurai (Shichinin no samurai),1954.0,Action|Adventure|Drama,13367,4.254769,4.233163
1173,12 Angry Men,1957.0,Drama,16569,4.243014,4.225767
2867,Fight Club,1999.0,Action|Crime|Drama|Thriller,58773,4.228311,4.223465
1164,One Flew Over the Cuckoo's Nest,1975.0,Drama,36058,4.218662,4.210907
42806,Band of Brothers,2001.0,Action|Drama|War,1356,4.398599,4.196711
891,Casablanca,1942.0,Drama|Romance,26890,4.206564,4.196388
